# 6-Dot Quantum Device GDS Layout Generator
## 量子点器件GDS布局生成器

本Notebook演示如何使用`SixDotQuantumDeviceGenerator`类来生成6量子点器件的GDS布局。

**功能包括：**
- 创建6量子点器件核心布局
- 生成Pad框架和引线
- 自动分层Z型扇出布线
- 分步可视化与GDS导出

In [ ]:
# 导入必要的库
import sys
import os
import gdstk
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

# 导入自定义类
from six_dot_generator import SixDotQuantumDeviceGenerator, plot_gds

# 设置输出目录
output_dir = "./output_gds"
os.makedirs(output_dir, exist_ok=True)
print(f"输出目录: {os.path.abspath(output_dir)}")

## 步骤1: 创建并可视化量子点器件核心

生成带有长引线的6量子点器件核心布局。

In [ ]:
# 初始化生成器
generator = SixDotQuantumDeviceGenerator(output_dir=output_dir)

# 生成量子点器件
lib_device, cell_device, device_points = generator.create_quantum_dot_device()

print("量子点器件已创建!")
print(f"器件点数: {len(device_points)}")
print("连接点标签:", list(device_points.keys()))

In [ ]:
# 可视化量子点器件
plot_gds(cell_device, title="步骤1: 6量子点器件核心 (带长引线)")

In [ ]:
# 保存步骤1的GDS
step1_path = os.path.join(output_dir, "step1_device_with_leads.gds")
lib_device.write_gds(step1_path)
print(f"步骤1 GDS已保存: {step1_path}")

## 步骤2: 创建并可视化Pad框架

生成带有引线和taper的焊盘框架。

In [ ]:
# 创建Pad框架
ACTIVE_SIZE = 200

lib_pads, cell_pads, pad_points, active_center, all_pads_info = generator.create_pad_frame(
    active_size=ACTIVE_SIZE,
    layout_width=1400,
    layout_height=1400,
    pad_width=100,
    pad_height=100,
    pad_spacing=30,
    edge_margin=50,
    trace_width=10,
    trace_spacing=10
)

print("Pad框架已创建!")
print(f"有源区中心: {active_center}")
print(f"Pad数量: {len(all_pads_info)}")

In [ ]:
# 可视化Pad框架
generator.plot_pad_frame(cell_pads, title="步骤2: Pad框架 (带引线)")

In [ ]:
# 保存步骤2的GDS
step2_path = os.path.join(output_dir, "step2_pad_frame.gds")
lib_pads.write_gds(step2_path)
print(f"步骤2 GDS已保存: {step2_path}")

## 步骤3: 组装器件和Pad框架

将量子点器件放置在Pad框架的有源区中心，但尚未布线。

In [ ]:
# 重新初始化生成器用于完整流程
generator = SixDotQuantumDeviceGenerator(output_dir=output_dir)

# 生成完整器件
lib_assembly, cell_assembly = generator.generate_full_device(active_size=ACTIVE_SIZE)

print("器件组装完成!")
print(f"活动中心: {generator.active_center}")
print(f"器件连接点数: {len(generator.device_points)}")
print(f"Pad连接点数: {len(generator.pad_points)}")

In [ ]:
# 可视化组装后的布局（未布线）
generator.plot_assembly(cell_assembly, title="步骤3: 器件已放置 (未布线)")

In [ ]:
# 保存步骤3的GDS
step3_path = os.path.join(output_dir, "step3_placed_unrouted.gds")
lib_assembly.write_gds(step3_path)
print(f"步骤3 GDS已保存: {step3_path}")

## 步骤4: 单线布线演示

演示使用分层Z型扇出算法连接单条线。

In [ ]:
# 选择一条线进行演示
label_to_connect = 'QD_PG4'

print(f"正在布设单条线: {label_to_connect}")

# 布设单条线
generator.route_all_connections(single_label=label_to_connect)

print(f"单条线 {label_to_connect} 已布设完成!")

In [ ]:
# 可视化单线布线结果
generator.plot_assembly(generator.cell, title=f"步骤4: 单线布线 ({label_to_connect})")

In [ ]:
# 保存步骤4的GDS
step4_path = os.path.join(output_dir, "step4_single_route.gds")
generator.lib.write_gds(step4_path)
print(f"步骤4 GDS已保存: {step4_path}")

## 步骤5: 完整布线

使用分层Z型扇出算法连接所有剩余的线。

In [ ]:
# 布设所有剩余的线
print("正在使用分层Z型扇出算法连接所有线...")
generator.route_all_connections()

print("所有线已布设完成!")

In [ ]:
# 可视化完整布线结果
generator.plot_assembly(generator.cell, title="步骤5: 最终完整布线")

In [ ]:
# 保存步骤5的GDS
step5_path = os.path.join(output_dir, "step5_fully_routed.gds")
generator.lib.write_gds(step5_path)
print(f"步骤5 GDS已保存: {step5_path}")

## 总结与输出文件列表

In [ ]:
# 列出所有生成的GDS文件
print("=" * 60)
print("生成的GDS文件列表:")
print("=" * 60)

gds_files = [
    ("step1_device_with_leads.gds", "量子点器件核心 (带长引线)"),
    ("step2_pad_frame.gds", "Pad框架 (带引线)"),
    ("step3_placed_unrouted.gds", "器件已放置 (未布线)"),
    ("step4_single_route.gds", "单线布线演示"),
    ("step5_fully_routed.gds", "最终完整布线")
]

for filename, description in gds_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"✓ {filename}")
        print(f"  描述: {description}")
        print(f"  大小: {size:,} bytes")
        print(f"  路径: {os.path.abspath(filepath)}")
        print()

print("=" * 60)
print("全部分步流程执行完毕!")
print("=" * 60)

## 高级用法：自定义参数

可以通过传递参数来自定义器件布局。以下是一些常用参数：

In [ ]:
# 示例：使用自定义参数创建器件
print("示例：使用自定义参数创建器件")
print("-" * 40)

# 自定义量子点器件参数
custom_qd_params = {
    'pg_max_width': 0.120,      # Plunger Gate 最大宽度
    'pg_vert_side_len': 0.040,  # Plunger Gate 垂直边长
    'pg_chamfer_h': 0.040,      # Plunger Gate 倒角高度
    'bg_max_width': 0.060,      # Barrier Gate 最大宽度
    'lead_width': 0.042,        # 引线宽度
    'lead_length_bot': 0.4,     # 底部引线长度
    'lead_length_top': 0.5,     # 顶部引线长度
    'sd_height': 0.10,         # S/D区域高度
    'sd_width': 0.35,           # S/D区域宽度
}

# 自定义Pad框架参数
custom_pad_params = {
    'layout_width': 1600,        # 布局宽度
    'layout_height': 1600,       # 布局高度
    'pad_width': 120,           # Pad宽度
    'pad_height': 120,          # Pad高度
    'pad_spacing': 40,          # Pad间距
    'active_size': 250,          # 有源区尺寸
    'trace_width': 12,           # 走线宽度
    'trace_spacing': 12,        # 走线间距
}

# 合并参数
all_params = {**custom_qd_params, **custom_pad_params}

print("自定义参数:")
for key, value in all_params.items():
    print(f"  {key}: {value}")

# 可以使用这些参数创建新的生成器
# generator2 = SixDotQuantumDeviceGenerator(output_dir="./custom_output")
# lib2, cell2 = generator2.generate_full_device(**all_params)

## 器件参数说明

### 量子点器件参数

| 参数 | 默认值 | 描述 |
|------|--------|------|
| `pg_max_width` | 0.120 | Plunger Gate 最大宽度 (μm) |
| `pg_vert_side_len` | 0.040 | Plunger Gate 垂直边长 (μm) |
| `pg_chamfer_h` | 0.040 | Plunger Gate 倒角高度 (μm) |
| `bg_max_width` | 0.060 | Barrier Gate 最大宽度 (μm) |
| `lead_width` | 0.042 | 引线宽度 (μm) |
| `lead_length_bot` | 0.4 | 底部引线长度 (μm) |
| `lead_length_top` | 0.5 | 顶部引线长度 (μm) |

### Pad框架参数

| 参数 | 默认值 | 描述 |
|------|--------|------|
| `layout_width` | 1400 | 布局宽度 (μm) |
| `layout_height` | 1400 | 布局高度 (μm) |
| `pad_width` | 100 | Pad宽度 (μm) |
| `pad_height` | 100 | Pad高度 (μm) |
| `active_size` | 180 | 有源区尺寸 (μm) |
| `trace_width` | 10 | 走线宽度 (μm) |
| `trace_spacing` | 10 | 走线间距 (μm) |

---

**输出文件位置:** `./output_gds/`

所有GDS文件已按步骤导出，可以在KLayout等EDA软件中打开查看。